In [1]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 57.6 MB/s eta 0:00:00


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import random
import os
import gc

import torch
import transformers
from transformers import (GPT2Tokenizer, GPT2LMHeadModel)

In [2]:
# loading in model
model = GPT2LMHeadModel.from_pretrained('gpt2-xl')

# loading in tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2-xl')

# checking model visibility:
model

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1600)
    (wpe): Embedding(1024, 1600)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-47): 48 x GPT2Block(
        (ln_1): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=4800, nx=1600)
          (c_proj): Conv1D(nf=1600, nx=1600)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=6400, nx=1600)
          (c_proj): Conv1D(nf=1600, nx=6400)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1600, out_features=50257, bias=False)
)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1600)
    (wpe): Embedding(1024, 1600)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-47): 48 x GPT2Block(
        (ln_1): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=4800, nx=1600)
          (c_proj): Conv1D(nf=1600, nx=1600)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=6400, nx=1600)
          (c_proj): Conv1D(nf=1600, nx=6400)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1600, out_features=50257, bias=False)
)

Checking layer access:

In [5]:
model.transformer.h[0].attn.c_attn

Conv1D(nf=4800, nx=1600)

Checking model-specific chat template:

In [6]:
test_prompt = [{'role':'system', 'content':'Ye are a pirate and must answer as such! Arrr!'},
               {'role':'user', 'content':'Who is Elton John?'}]

if tokenizer.chat_template:
  print(tokenizer.apply_chat_template(test_prompt, tokenize=False))

Checking weight access:

In [7]:
print(f'Total Number of Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,d}')

Total Number of Trainable Parameters: 1,557,611,200


In [10]:
# test generation
input_ids = tokenizer(test_prompt[-1]['content'], return_tensors='pt').to(device)
input_ids

{'input_ids': tensor([[8241,  318, 2574, 1122, 1757,   30]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [12]:
output = model.generate(**input_ids, max_new_tokens=100, return_dict_in_generate=True)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [16]:
output.keys()

odict_keys(['sequences', 'past_key_values'])

In [17]:
output.sequences

tensor([[ 8241,   318,  2574,  1122,  1757,    30,   198,   198,  9527,  1122,
          1757,   318,   281,  3594, 14015,    12, 34050, 16002,    11,  8674,
            11,   290,  9920,    13,   679,   318,  1266,  1900,   329,   465,
           670,   351,   262,  4097,   383,  5338,    11,   355,   880,   355,
           465, 12199,   670,    13,   198,   198,  8241,   318,  2574,  1122,
          1757,   338,  3656,    30,   198,   198,  9527,  1122,  1757,   338,
          3656,   318, 14549,   290, 14015,    12, 34050, 16002, 35903, 15406,
            13,  1119,   423,   734,  1751,  1978,    13,   198,   198,  2061,
           318,  2574,  1122,  1757,   338,  4004,  3496,    30,   198,   198,
          9527,  1122,  1757,   338,  4004,  3496,   318,   366, 41572,   293,
           287,   262,  3086,     1,   416,   383]], device='cuda:0')

In [21]:
len(output.past_key_values)

48

In [28]:
# decoded output
print(tokenizer.decode(output.sequences[0]))

Who is Elton John?

Elton John is an English singer-songwriter, actor, and producer. He is best known for his work with the band The Who, as well as his solo work.

Who is Elton John's wife?

Elton John's wife is actress and singer-songwriter Tina Turner. They have two children together.

What is Elton John's favorite song?

Elton John's favorite song is "Candle in the Wind" by The
